## Note on Differencing (Stationarity)
**Question:** Should we manually difference the data (e.g., $y_t - y_{t-1}$) before fitting?

**Answer:** No. Time differencing is handled automatically by the ARIMA model parameters ($d$ for trend, $D$ for seasonality). 
1. `auto_arima` determines the optimal $d$ and $D$ values.
2. `SARIMAX` applies this differencing internally during fitting.
3. `SARIMAX` automatically "integrates" (reverses the differencing) during prediction, so the output forecasts are on the original scale.

Manual differencing would require complex post-processing to reconstruct the scale, especially in a recursive loop. We rely on the model's built-in handling.

# AutoARIMA Test Pipeline
This notebook adapts the LSTM pipeline for AutoARIMA, using the same recursive inference and validation logic.

In [ ]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error
import os

In [ ]:
# -----------------------------------------------------------------------------
# DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------
DATA_PATH = '../dataset/subset_set.feather'  # Adjust path if needed
print(f"Loading data from {DATA_PATH}...")
# Use pd.read_feather instead of feather.read_table to get a DataFrame directly
df = pd.read_feather(DATA_PATH)

# Define constants
DATE_COL = 'date'
TARGET_COL = 'value'  # Assuming 'value' is the target variable (sales at the end of the day)


# Handle DATE_COL (ensure it is a column and not in the index)
if DATE_COL in df.index.names:
    if DATE_COL in df.columns:
        # If it's in both, drop the index version to avoid "cannot insert" error
        df = df.reset_index(drop=True)
    else:
        # If it's only in the index, move it to a column
        df = df.reset_index()

# If DATE_COL is NOT in columns (and wasn't in index), we have a problem, but assuming it exists somewhere.
# Just to be safe, if we still have a complex index, reset it.
if df.index.name == DATE_COL:
     df = df.reset_index(drop=True)
     
# Fallback: simple reset to ensure RangeIndex 0..N
df = df.reset_index(drop=True)

# Ensure date is datetime and sort
df[DATE_COL] = pd.to_datetime(df[DATE_COL])
df = df.sort_values(DATE_COL)
df = df.reset_index(drop=True) # Final clean reset

# Prepare target variable
# Using 'value' as the target (sales at the end of the day)
y = df[TARGET_COL]

print(len(y))


# -----------------------------------------------------------------------------
# FEATURE ENGINEERING (EXOGENOUS VARIABLES)
# -----------------------------------------------------------------------------
# 1. Day of week (0=Monday, 6=Sunday)
df['day_of_week'] = df[DATE_COL].dt.dayofweek

# 2. Month (1=January, 12=December)
df['month'] = df[DATE_COL].dt.month

# 3. Day of month (1-31) - captures paydays
df['day_of_month'] = df[DATE_COL].dt.day

# 4. Is Weekend (Binary)
df['is_weekend'] = (df['day_of_week'] >= 5).astype(float)

# Christmas Day flag (1 on Dec 25, else 0)
df['is_christmas_day'] = ((df[DATE_COL].dt.month == 12) & (df[DATE_COL].dt.day == 25)).astype(float)

# 5. Promotions
# Identify all columns that start with 'promo_'
promo_cols = [col for col in df.columns if col.startswith('promo_')]
print(f"Found {len(promo_cols)} promotion columns: {promo_cols}")

# Ensure promo columns are numeric (float)
for col in promo_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype(float)

# Define the list of exogenous features to use
#EXOG_COLS = None
EXOG_COLS = ['day_of_week', 'month', 'day_of_month', 'is_weekend', 'is_christmas_day'] + promo_cols


# -----------------------------------------------------------------------------
# DATA SPLITTING
# -----------------------------------------------------------------------------
# Define split sizes
train_size = 455
val_size = 153
forecast_horizon = 153

lookback_window = 30


df.head()

In [ ]:
# DATA SPLITTING
train_size = 455
val_size = 153
forecast_horizon = 153
lookback_window = 30
print(df[TARGET_COL].describe())

In [ ]:
# -----------------------------------------------------------------------------
# SEASONALITY ANALYSIS STAGE
# -----------------------------------------------------------------------------
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

def analyze_seasonality(df, target_col, item_id, store_id, max_lag=60):
    """
    Plots ACF and PACF to help determine the optimal 'm' parameter.
    """
    subset = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
    if DATE_COL in subset.columns:
        subset = subset.sort_values(DATE_COL)
    
    ts = subset[target_col].dropna()
    
    if len(ts) < max_lag:
        print(f"Not enough data for item {item_id} store {store_id}")
        return

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    
    # ACF Plot
    plot_acf(ts, lags=max_lag, ax=axes[0])
    axes[0].set_title(f"ACF (Autocorrelation) - Item {item_id}, Store {store_id}")
    axes[0].set_xlabel("Lag (Days)")
    
    # PACF Plot
    plot_pacf(ts, lags=max_lag, ax=axes[1])
    axes[1].set_title(f"PACF (Partial Autocorrelation)")
    axes[1].set_xlabel("Lag (Days)")
    
    plt.show()
    print("Interpretation Guide:")
    print("- Spike at lag 7, 14, 21... -> Strong Weekly Seasonality (m=7)")
    print("- Spike at lag 30, 60...    -> Strong Monthly Seasonality (m=30)")
    print("- No clear pattern          -> Consider seasonal=False or m=1")

target_products = []
if len(target_products) > 0:
    products = df[df['item_id'].isin(target_products)][['item_id', 'store_id']].drop_duplicates().values
else:
    products = df[['item_id', 'store_id']].drop_duplicates().values
results = []
os.makedirs('grid_search_plots', exist_ok=True)
# Run analysis for the first target product
if len(products) > 0:
    first_item = products[0][0]
    # Find a store that has this item
    first_store = products[0][1]
    analyze_seasonality(df, TARGET_COL, first_item, first_store)
else:
    print("No target products defined.")

In [ ]:
from statsmodels.tsa.stattools import acf

def get_seasonal_candidates(df, target_col, item_id, store_id, max_lag=35):
    """
    Analyzes ACF to automatically suggest 'm' candidates (e.g., 7 for weekly, 30 for monthly).
    Returns a list of integer candidates.
    Matches standard periods (7, 30) if they are statistically significant.
    """
    subset = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
    if DATE_COL in subset.columns:
        subset = subset.sort_values(DATE_COL)
    
    ts = subset[target_col].dropna().values
    n = len(ts)
    if n < max_lag:
        return [1] # Not enough data

    # Calculate ACF
    # nlags=max_lag returns lags 0..max_lag
    acf_values = acf(ts, nlags=max_lag, fft=True)
    
    # Significance threshold (approx 95% confidence interval for white noise)
    threshold = 1.96 / np.sqrt(n)
    
    candidates = [1]
    
    # Check specifically for Weekly (7) and Monthly (30)
    # We prioritize 7 and 30 because they are structural seasonalities.
    # We avoid picking random lags (like 2 or 3) as "seasonality" because those are usually covered by AR terms.
    
    potential_periods = [7, 30] 
    
    for p in potential_periods:
        if p <= max_lag:
             # Check if the lag itself is significant
            if abs(acf_values[p]) > threshold:
                candidates.append(p)
    
    # If no structural seasonality found, fallback to 1 (non-seasonal)
    if not candidates:
        candidates = [1]
        
    return candidates

# AutoARIMA Experiment Function
This function matches the LSTM pipeline: same splits, recursive inference, and validation.

In [ ]:

'''
def autoarima_experiment_grid(df, target, item_id, store_id, train_size=432, val_size=153, forecast_window=153, lookback_window=30, save_plot_path=None):
    """
    Run AutoARIMA with recursive forecasting using a lookback window.
    """
    total_train_val = train_size + val_size
    train_slice = slice(-(total_train_val+forecast_window), -forecast_window)
    test_slice = slice(-forecast_window, None)
    train = df[target][train_slice].values
    test = df[target][test_slice].values

    # Fit model on train set
    model = auto_arima(train, 
                       start_p=0, d=None, start_q=2, 
                       max_p=5, max_d=2, max_q=5, start_P=1, D=None, start_Q=1, 
                       max_P=2, max_D=1, max_Q=2, seasonal=False, 
                       stepwise=True, suppress_warnings=True)

    # Recursive forecast using lookback window
    history = list(train[-lookback_window:])  # Start with last lookback_window values from train
    forecast = []
    for i in range(forecast_window):
        # Fit ARIMA on the current history window
        model_update = ARIMA(history, order=model.order).fit()
        next_pred = model_update.forecast(steps=1)[0]
        forecast.append(next_pred)
        history.append(next_pred)
        # Keep only the last lookback_window values
        if len(history) > lookback_window:
            history = history[-lookback_window:]

    forecast = np.array(forecast)
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mae = mean_absolute_error(test, forecast)

    # Save all plots in grid_search_plots directory
    plot_dir = f'grid_search_plots'
    os.makedirs(plot_dir, exist_ok=True)
    plot_filename = f'{plot_dir}/autoarima_item{item_id}_store{store_id}.png'

    if save_plot_path:
        plt.figure(figsize=(12,6))
        plt.plot(range(len(train)), train, label='Train')
        plt.plot(range(len(train), len(train)+len(test)), test, label='Test')
        plt.plot(range(len(train), len(train)+len(test)), forecast, label='Forecast')
        plt.title(f'AutoARIMA Recursive Forecast (Item={item_id}, Store={store_id})')
        plt.legend()
        plt.savefig(plot_filename)
        plt.close()

    return rmse, mae, model.order, plot_filename
'''

In [ ]:
def autoarima_fit_once_lookback_state_forecast(
    df, target, item_id, store_id,
    train_size=455, val_size=153,
    forecast_window=153,
    lookback_window=30,
    date_col=None,                 # optional, only for sorting/plots
    save_plot_path=None,
    seasonal=False, m=1,           # set seasonal=True, m=7 for daily weekly seasonality if desired
    exog_cols=None                 # List of exogenous column names
):
    """
    APPROACH 1: "Rolling / Lookback State"
    1) Use auto_arima to select (order, seasonal_order) for the specific 'm' provided.
    2) Fit statsmodels SARIMAX ONCE on train to learn parameters.
    3) Recursively forecast 1 step at a time, using ONLY the last 'lookback_window' observations 
       to update the state (mimicking an LSTM sliding window).
    """

    dfp = df.copy()
    if date_col is not None:
        dfp[date_col] = pd.to_datetime(dfp[date_col])
        dfp = dfp.sort_values(date_col).reset_index(drop=True)
    else:
        dfp = dfp.reset_index(drop=True)

    total_train_val = train_size + val_size
    train_slice = slice(-(total_train_val + forecast_window), -forecast_window)
    test_slice  = slice(-forecast_window, None)

    y_train = dfp[target].iloc[train_slice].astype(float).values
    y_test  = dfp[target].iloc[test_slice].astype(float).values

    # Handle Exogenous Variables
    X_train = None
    X_test = None
    if exog_cols:
        X_train = dfp[exog_cols].iloc[train_slice].astype(float).values
        X_test = dfp[exog_cols].iloc[test_slice].astype(float).values
    
    # If m=1, force seasonal=False for this run
    current_seasonal = True if (seasonal and m > 1) else False

    try:
        # Using auto_arima to SEARCH for parameters
        best_model = auto_arima(
            y_train,
            X=X_train, 
            seasonal=current_seasonal, 
            m=m if current_seasonal else 1, 
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore",
            start_p=0, start_q=2, max_p=5, max_q=5, max_d=2,
            start_P=1, start_Q=1, max_P=2, max_Q=2, max_D=1
        )
    except Exception as e:
        print(f"  m={m} failed: {e}")
        raise ValueError(f"AutoARIMA failed for m={m}")

    # Use the best parameters found
    order = tuple(best_model.order)
    seasonal_order = tuple(best_model.seasonal_order)
    best_aic = best_model.aic()
    
    # Validation Print:
    print(f"Selected best model: Order={order}, Seasonal={seasonal_order} (AIC={best_aic:.2f})")
    
    # --- 2) Fit ONCE with statsmodels, parameters estimated on full y_train ---
    # We pass 'seasonal_order' directly, which contains the 'm' found by auto_arima.
    
    # Increase maxiter to avoid convergence warnings
    model = SARIMAX(
        y_train,
        exog=X_train, 
        order=order,
        seasonal_order=seasonal_order, # Important: This passes the (P,D,Q,m) tuple
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    # fit() arguments: maxiter increased to help convergence
    res = model.fit(disp=False, maxiter=200, method='lbfgs')
    params = res.params

    # --- 3) Recursive Forecast with Sliding Window (LSTM-style) ---
    # We initialize history with the end of training data
    history = list(y_train[-lookback_window:])

    # Also need history of exogenous variables if used
    if exog_cols:
        history_exog = list(X_train[-lookback_window:])
        
    forecast = []

    # Loop for each forecast step
    for i in range(forecast_window):
        # Window for this step: strictly the last 'lookback_window' observations
        current_data = np.asarray(history[-lookback_window:], dtype=float)

        if exog_cols:
            current_exog = np.asarray(history_exog[-lookback_window:], dtype=float)
            next_exog_step = np.asarray(X_test[i:i+1], dtype=float)  # (1, k)
        else:
            current_exog = None
            next_exog_step = None
        try:
            # Re-initialize model structure on the NEW window
            mod_step = SARIMAX(
                current_data,
                exog=current_exog, # Exog for the history window
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False
            )
            # Apply parameters to this window to update state
            # No fit() called here, just filter(), so no convergence issue in loop usually
            res_step = mod_step.filter(params)
            
            # Forecast the single next step (t+1)
            pred = res_step.forecast(steps=1, exog=next_exog_step)[0] # future exog
        except Exception as e:
            # Fallback for edge cases
            pred = history[-1]
        
        #if pred < 0:
        #    pred = 0.0
            
        forecast.append(pred)
        history.append(pred)

        if exog_cols:
             # Append the used future exog to history so it becomes part of the window for next step
             history_exog.append(X_test[i])

    forecast = np.asarray(forecast)
    #forecast = np.maximum(forecast, 0) 
    rmse = float(np.sqrt(mean_squared_error(y_test, forecast)))
    mae  = float(mean_absolute_error(y_test, forecast))

    return rmse, mae, order, seasonal_order, forecast, y_train, y_test

In [ ]:
def autoarima_fit_once_full_history_forecast(
    df, target, item_id, store_id,
    train_size=432, val_size=153,
    forecast_window=153,
    save_plot_path=None,
    seasonal=False, m=1,
    exog_cols=None # List of exogenous column names
):
    """
    APPROACH 2: "Standard / Full History"
    1) Use auto_arima to select order for the specific 'm' provided.
    2) Fit statsmodels SARIMAX ONCE on train.
    3) Forecast the entire horizon using the full accumulated history (no sliding window truncation).
    """
    dfp = df.copy()
    if date_col := 'date' in dfp.columns:
         dfp['date'] = pd.to_datetime(dfp['date'])
         dfp = dfp.sort_values('date').reset_index(drop=True)
    else:
         dfp = dfp.reset_index(drop=True)

    total_train_val = train_size + val_size
    train_slice = slice(-(total_train_val + forecast_window), -forecast_window)
    test_slice  = slice(-forecast_window, None)
    
    y_train = dfp[target].iloc[train_slice].astype(float).values
    y_test  = dfp[target].iloc[test_slice].astype(float).values

    # Handle Exogenous Variables
    X_train = None
    X_test = None
    if exog_cols:
        X_train = dfp[exog_cols].iloc[train_slice].astype(float).values
        X_test = dfp[exog_cols].iloc[test_slice].astype(float).values

    # --- 1) Choose order via auto_arima (candidate selection) ---
    print(f"Testing m={m}...")
    
    # If m=1, force seasonal=False for this run unless specifically requested otherwise
    current_seasonal = True if (seasonal and m > 1) else False
    
    try:
        # Use auto_arima to find best (p,d,q) for this specific m
        best_model = auto_arima(
            y_train,
            X=X_train, # Pass exog variables
            seasonal=current_seasonal, 
            m=m if current_seasonal else 1,
            stepwise=True,
            suppress_warnings=True,
            error_action="ignore",
            start_p=0, start_q=2, max_p=5, max_q=5, max_d=2,
            start_P=1, start_Q=1, max_P=2, max_Q=2, max_D=1
        )
    except Exception as e:
        print(f"  m={m} failed: {e}")
        raise ValueError(f"AutoARIMA failed for m={m}")
    
    order = tuple(best_model.order)
    seasonal_order = tuple(best_model.seasonal_order)
    best_aic = best_model.aic()
    
    print(f"Selected best model: Order={order}, Seasonal={seasonal_order} (AIC={best_aic:.2f})")

    
    # 2) Fit statsmodels
    model = SARIMAX(
        y_train,
        exog=X_train, # Pass exog variables
        order=order,
        seasonal_order=seasonal_order,
        enforce_stationarity=False,
        enforce_invertibility=False
    )
    # Increase maxiter to avoid convergence warnings
    res = model.fit(disp=False, maxiter=200, method='lbfgs')
    
    # 3) Forecast (Standard)
    # This uses the full state from y_train to predict steps 1..N
    # .forecast(steps=N) returns the point forecasts directly.
    forecast = res.forecast(steps=forecast_window, exog=X_test) # Pass future exog
    #forecast = np.maximum(forecast, 0) 
    rmse = float(np.sqrt(mean_squared_error(y_test, forecast)))
    mae  = float(mean_absolute_error(y_test, forecast))
    
    return rmse, mae, order, seasonal_order, forecast

# Grid Search Loop for Products and Seeds
This cell runs AutoARIMA for each product and seed, saving results and plots.

In [ ]:

for item_id, store_id in products:
    print(f"\nProcessing Item {item_id} Store {store_id}...")
    df_product = df[(df['item_id'] == item_id) & (df['store_id'] == store_id)].copy()
    if DATE_COL in df_product.columns:
        df_product[DATE_COL] = pd.to_datetime(df_product[DATE_COL])
        df_product = df_product.sort_values(DATE_COL).reset_index(drop=True)
    else:
         df_product = df_product.reset_index(drop=True)

    # --- Automatic Seasonality Detection ---
    seasonal_candidates = get_seasonal_candidates(
        df=df_product,
        target_col=TARGET_COL,
        item_id=item_id,
        store_id=store_id,
        max_lag=1 # Checking up to monthly
    )
    
    print(f"  Selected Seasonality Candidates (m): {seasonal_candidates}")

    # Iterate over EACH candidate to test them individually and generate separate plots
    for m_val in seasonal_candidates:
        print(f"  >> Testing candidate m={m_val}")
        
        # --- Run Approach 1: Lookback (Recursive with sliding window) ---
        rmse_lb, mae_lb, order_lb, seasonal_order_lb, forecast_lb, y_train, y_test = autoarima_fit_once_lookback_state_forecast(
            df=df_product,
            target=TARGET_COL,
            item_id=item_id,
            store_id=store_id,
            train_size=train_size,
            val_size=val_size,
            forecast_window=forecast_horizon,
            lookback_window=30,  
            seasonal=True if m_val > 1 else False, 
            m=m_val, # Pass single m
            exog_cols=EXOG_COLS 
        )
        
        # --- Run Approach 2: Full History (Standard Recursive) ---
        rmse_full, mae_full, order_full, seasonal_order_full, forecast_full = autoarima_fit_once_full_history_forecast(
            df=df_product,
            target=TARGET_COL,
            item_id=item_id,
            store_id=store_id,
            train_size=train_size,
            val_size=val_size,
            forecast_window=forecast_horizon,
            seasonal=True if m_val > 1 else False, 
            m=m_val, # Pass single m
            exog_cols=EXOG_COLS 
        )

        # --- Compare Plot for this specific m ---
        plot_filename = f'grid_search_plots/comparison_item{item_id}_store{store_id}_m{m_val}.png'
        plt.figure(figsize=(14, 7))
        
        # Plot Training Data (last part only for clarity)
        plt.plot(range(len(y_train)), y_train, label='Train', color='gray', alpha=0.5)
        
        # Plot Test Data
        test_range = range(len(y_train), len(y_train) + len(y_test))
        plt.plot(test_range, y_test, label='Actual Test', color='black', linewidth=2)
        
        # Plot Lookback Forecast
        plt.plot(test_range, forecast_lb, label=f'Lookback (m={m_val})\nRMSE={rmse_lb:.2f}, MAE={mae_lb:.2f}', color='blue', linestyle='--')
        
        # Plot Full History Forecast
        plt.plot(test_range, forecast_full, label=f'FullHist (m={m_val})\nRMSE={rmse_full:.2f}, MAE={mae_full:.2f}', color='red', linestyle='--')
        
        plt.title(f'Comparison (m={m_val}): Lookback vs Full History (Item={item_id})')
        plt.legend()
        
        # Add text box with metrics
        stats_text = (
            f"m={m_val}\n"
            f"Lookback: RMSE={rmse_lb:.2f}, MAE={mae_lb:.2f}\n"
            f"FullHist: RMSE={rmse_full:.2f}, MAE={mae_full:.2f}"
        )
        plt.gcf().text(0.02, 0.02, stats_text, fontsize=10, bbox=dict(facecolor='white', alpha=0.8))

        plt.savefig(plot_filename)
        plt.close()
        print(f"    Saved plot to {plot_filename}")
        
        results.append({
            'item_id': item_id,
            'store_id': store_id,
            'tested_m': m_val,
            'lookback_rmse': rmse_lb,
            'lookback_mae': mae_lb,
            'lookback_order': order_lb,
            'lookback_seasonal_order': seasonal_order_lb,
            'full_rmse': rmse_full,
            'full_mae': mae_full,
            'full_order': order_full,
            'full_seasonal_order': seasonal_order_full,
            'plot_path': plot_filename
        })

results_df = pd.DataFrame(results)
results_df.to_csv('autoarima_comparison_results.csv', index=False)
print("Calibration complete. Results saved to 'autoarima_comparison_results.csv'.")